# STEMData — Construcción del modelo


In [ ]:
import pandas as pd

df = pd.read_csv("stemdata_limpio.csv")

print(f"Filas: {df.shape[0]} — Columnas: {df.shape[1]}")
df.head()

In [ ]:
df.columns.tolist()

## Capa 1: Brecha STEM

Combinamos los dos puntajes en un solo indicador, y lo comparamos contra el promedio nacional.


In [ ]:
df['puntaje_stem'] = (df['promedio_ciencias'] + df['promedio_matematicas']) / 2

df[['municipio', 'zona', 'puntaje_stem']].head()

### Calcular la brecha contra el promedio nacional

In [ ]:
promedio_nacional = df['puntaje_stem'].mean()
df['brecha_stem'] = df['puntaje_stem'] - promedio_nacional

print(f"Promedio nacional STEM: {promedio_nacional:.2f}")
df[['municipio', 'zona', 'puntaje_stem', 'brecha_stem']].head()

### Clasificar según qué tan lejos está del promedio



In [ ]:
desviacion_stem = df['brecha_stem'].std()

def clasificar_brecha(brecha):
    if brecha < -desviacion_stem:
        return 'crítico'
    elif brecha < 0:
        return 'alto'
    elif brecha < desviacion_stem:
        return 'medio'
    else:
        return 'bajo'

df['nivel_brecha'] = df['brecha_stem'].apply(clasificar_brecha)

df['nivel_brecha'].value_counts()

## Capa 2: Viabilidad tecnológica (medición objetiva de conectividad)

### Clasificar conectividad por desviación estándar

In [ ]:
promedio_conectividad = df['total_accesos_internet'].mean()
desviacion_conectividad = df['total_accesos_internet'].std()

def clasificar_conectividad(valor):
    if valor < promedio_conectividad - desviacion_conectividad:
        return 'desconectado'
    elif valor < promedio_conectividad:
        return 'básico'
    elif valor < promedio_conectividad + desviacion_conectividad:
        return 'medio'
    else:
        return 'alto'

df['nivel_conectividad_base'] = df['total_accesos_internet'].apply(clasificar_conectividad)

df[['municipio', 'zona', 'total_accesos_internet', 'nivel_conectividad_base']].head()

In [ ]:
df['zona'].unique()

In [ ]:
df['nivel_conectividad'] = df['nivel_conectividad_base']

df[['municipio', 'zona', 'total_accesos_internet', 'nivel_conectividad']].head(10)

### Árbol de decisión visual



In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

X = df[['total_accesos_internet']]
y = df['nivel_conectividad']

arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X, y)

plt.figure(figsize=(14, 6))
plot_tree(arbol, feature_names=['total_accesos_internet'], class_names=arbol.classes_, filled=True)
plt.show()

## Capa 3: Generador de planeación STEM+



In [ ]:
import unicodedata
import re

def quitar_tildes(texto):
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r'[.,]', '', texto)   # quita puntos y comas (ej. "BOGOTA, D.C." -> "BOGOTA DC")
    texto = re.sub(r'\s+', ' ', texto)   # colapsa espacios dobles en uno
    return texto.strip()

def generar_planeacion(nombre_municipio, zona_deseada):
    municipio_buscado = quitar_tildes(nombre_municipio)
    zona_buscada = quitar_tildes(zona_deseada)

    # Acepta "URBANA", "URBANO", etc. -- se queda solo con la raíz "URBAN"
    if zona_buscada.startswith("URBAN"):
        zona_buscada = "URBANO"
    elif zona_buscada.startswith("RURAL"):
        zona_buscada = "RURAL"

    filtro = (
        (df['municipio'].apply(quitar_tildes) == municipio_buscado) &
        (df['zona'].apply(quitar_tildes) == zona_buscada)
    )
    fila = df[filtro]

    if fila.empty:
        return f"No se encontró '{nombre_municipio}' en zona '{zona_deseada}' en la base de datos."

    fila = fila.iloc[0]

    brecha = fila['nivel_brecha']
    conectividad = fila['nivel_conectividad']
    zona = fila['zona'].strip().upper()

    # La conectividad decide si la actividad puede apoyarse en herramientas digitales
    if conectividad in ['desconectado', 'básico']:
        actividad = "Actividad totalmente desconectada, con materiales físicos o reciclables."
    else:
        actividad = "Actividad híbrida, combinando materiales físicos con alguna herramienta digital básica."

    # La zona decide de dónde vienen esos materiales y cómo se organiza la logística
    if zona.startswith('RURAL'):
        logistica = "Prioriza materiales disponibles en el entorno inmediato (elementos del campo, reciclables, herramientas caseras), pensando en tiempos de desplazamiento y acceso limitado a papelerías o tiendas especializadas."
    else:
        logistica = "Pueden complementarse con materiales de papelería o ferretería cercana, y es más viable coordinar salidas cortas o recursos compartidos entre colegios de la misma zona."

    if brecha in ['crítico', 'alto']:
        enfoque = "Enfoque de refuerzo: actividad guiada paso a paso, con acompañamiento cercano del docente."
    else:
        enfoque = "Enfoque de profundización: actividad más abierta, con espacio para exploración autónoma."

    reporte = f"""
    Municipio: {fila['municipio']} ({fila['zona']})
    Nivel de brecha STEM: {brecha}
    Nivel de conectividad: {conectividad}

    Actividad sugerida: {actividad}
    Consideración logística: {logistica}
    Enfoque pedagógico sugerido: {enfoque}
    """
    return reporte

Visualización para el docente

### Gráfico A — Tu municipio vs. el promedio nacional (puntaje STEM)


In [ ]:
def graficar_comparacion_stem(nombre_municipio, zona_deseada):
    municipio_buscado = quitar_tildes(nombre_municipio)
    zona_buscada = quitar_tildes(zona_deseada)
    if zona_buscada.startswith("URBAN"):
        zona_buscada = "URBANO"
    elif zona_buscada.startswith("RURAL"):
        zona_buscada = "RURAL"

    filtro = (
        (df['municipio'].apply(quitar_tildes) == municipio_buscado) &
        (df['zona'].apply(quitar_tildes) == zona_buscada)
    )
    fila = df[filtro]

    if fila.empty:
        print(f"No se encontró '{nombre_municipio}' en zona '{zona_deseada}'.")
        return

    puntaje_municipio = fila.iloc[0]['puntaje_stem']
    promedio_nacional_actual = df['puntaje_stem'].mean()

    etiquetas = [f"{nombre_municipio.title()}\n({zona_deseada.title()})", "Promedio\nnacional"]
    valores = [puntaje_municipio, promedio_nacional_actual]
    colores = ['#27ae60' if puntaje_municipio >= promedio_nacional_actual else '#c0392b', '#7f8c8d']

    plt.figure(figsize=(6, 5))
    plt.bar(etiquetas, valores, color=colores)
    plt.title('Tu municipio vs. el promedio nacional (puntaje STEM)')
    plt.ylabel('Puntaje STEM')
    plt.show()

graficar_comparacion_stem("MEDELLIN", "URBANA")

### Gráfico B — Dónde se ubica tu municipio en la distribución nacional de conectividad

In [ ]:
def graficar_ubicacion_conectividad(nombre_municipio, zona_deseada):
    municipio_buscado = quitar_tildes(nombre_municipio)
    zona_buscada = quitar_tildes(zona_deseada)
    if zona_buscada.startswith("URBAN"):
        zona_buscada = "URBANO"
    elif zona_buscada.startswith("RURAL"):
        zona_buscada = "RURAL"

    filtro = (
        (df['municipio'].apply(quitar_tildes) == municipio_buscado) &
        (df['zona'].apply(quitar_tildes) == zona_buscada)
    )
    fila = df[filtro]

    if fila.empty:
        print(f"No se encontró '{nombre_municipio}' en zona '{zona_deseada}'.")
        return

    accesos_municipio = fila.iloc[0]['total_accesos_internet']

    plt.figure(figsize=(8, 5))
    plt.hist(df['total_accesos_internet'], bins=40, color='#bdc3c7')
    plt.axvline(accesos_municipio, color='#c0392b', linewidth=2.5, label=f"{nombre_municipio.title()}")
    plt.title('Dónde se ubica tu municipio en conectividad nacional')
    plt.xlabel('Total de accesos a internet')
    plt.ylabel('Cantidad de municipios')
    plt.xscale('log')
    plt.legend()
    plt.show()

graficar_ubicacion_conectividad("MEDELLIN", "URBANA")

### Función que el agente llamaría en cada consulta



In [ ]:
def consultar_municipio(nombre_municipio, zona_deseada):
    print(generar_planeacion(nombre_municipio, zona_deseada))
    graficar_comparacion_stem(nombre_municipio, zona_deseada)
    graficar_ubicacion_conectividad(nombre_municipio, zona_deseada)

consultar_municipio("MEDELLIN", "RURAL")

## Agregar coordenadas y exportar datos para el mapa del prototipo



In [ ]:
url_coordenadas = "https://www.datos.gov.co/api/views/gdxc-w37w/rows.csv?accessType=DOWNLOAD"

coordenadas_crudo = pd.read_csv(url_coordenadas)

print(f"Filas: {coordenadas_crudo.shape[0]} — Columnas: {coordenadas_crudo.shape[1]}")
coordenadas_crudo.columns.tolist()

In [ ]:
COLUMNA_MUNICIPIO = "Nombre Municipio"
COLUMNA_LATITUD = "Latitud"
COLUMNA_LONGITUD = "longitud"

coordenadas = coordenadas_crudo[[COLUMNA_MUNICIPIO, COLUMNA_LATITUD, COLUMNA_LONGITUD]].copy()
coordenadas.columns = ["municipio", "latitud", "longitud"]

# Corregir la coma decimal (texto "-75,581775" -> número -75.581775)
coordenadas["latitud"] = coordenadas["latitud"].astype(str).str.replace(",", ".").astype(float)
coordenadas["longitud"] = coordenadas["longitud"].astype(str).str.replace(",", ".").astype(float)

coordenadas["municipio"] = coordenadas["municipio"].apply(quitar_tildes)
coordenadas["municipio"] = coordenadas["municipio"].replace({"BOGOTA DC": "BOGOTA"})
coordenadas = coordenadas.dropna().drop_duplicates(subset="municipio")

print(f"Municipios con coordenadas: {coordenadas.shape[0]}")
coordenadas.head()

### Cruzar las coordenadas con la tabla del modelo

In [ ]:
df['municipio_normalizado'] = df['municipio'].apply(quitar_tildes)

df_con_mapa = pd.merge(
    df,
    coordenadas,
    left_on='municipio_normalizado',
    right_on='municipio',
    how='left',
    suffixes=('', '_coord')
)

print(f"Filas con coordenadas encontradas: {df_con_mapa['latitud'].notna().sum()} de {df_con_mapa.shape[0]}")
df_con_mapa[['municipio', 'zona', 'latitud', 'longitud']].head(10)

### Archivo JSON

In [ ]:
import json

registros = []
for _, fila in df_con_mapa.iterrows():
    if pd.isna(fila['latitud']):
        continue
    registros.append({
        "municipio": fila['municipio'],
        "zona": fila['zona'],
        "lat": round(float(fila['latitud']), 5),
        "lon": round(float(fila['longitud']), 5),
        "promedio_ciencias": round(float(fila['promedio_ciencias']), 1),
        "promedio_matematicas": round(float(fila['promedio_matematicas']), 1),
        "puntaje_stem": round(float(fila['puntaje_stem']), 1),
        "nivel_brecha": fila['nivel_brecha'],
        "total_accesos_internet": int(fila['total_accesos_internet']),
        "nivel_conectividad": fila['nivel_conectividad'],
    })

with open('stemdata_prototipo.json', 'w', encoding='utf-8') as f:
    json.dump(registros, f, ensure_ascii=False, indent=2)

print(f"Municipios exportados: {len(registros)}")

from google.colab import files
files.download('stemdata_prototipo.json')

## Resultado final (CSV para el repositorio)


In [ ]:
df.to_csv('stemdata_con_modelo.csv', index=False, encoding='utf-8-sig')

from google.colab import files
files.download('stemdata_con_modelo.csv')

print("Archivo generado y descargado ✅")